# Astrodynamics - Module d'atterrissage Eagle 1
## Installation & Exploration des espaces d'observation et d'actions

In [1]:
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium.wrappers import RecordVideo
import mlflow
import os

env_name = 'LunarLander-v3'
video_folder = "./videos_dqn"
log_dir='./logs_dqn'
SEED=42

train_env = gym.make(env_name)
train_env.reset(seed=SEED)
train_env.action_space.seed(SEED)

video_env = gym.make(env_name, render_mode="rgb_array")
video_env.reset(seed=SEED)
video_env.action_space.seed(SEED)
video_env = RecordVideo(
    video_env,
    video_folder=video_folder,
    episode_trigger=lambda ep_id: True,  # enregistre tous les épisodes
    name_prefix="dqn_lunarlander"
)

# Afficher les espaces d'observation et d'action
print("Espace d'observation :", train_env.observation_space)
print("Espace d'action :", train_env.action_space)

# Obtenir dynamiquement les dimensions de ces espaces
states_dim = train_env.observation_space.shape[0]
actions_dim = train_env.action_space.n

print("Espace d'observation - dimensions :",states_dim)
print("Espace d'action - dimensions :", actions_dim)




/usr/local/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /workspace/src/mission/videos_dqn folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Espace d'observation : Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Espace d'action : Discrete(4)
Espace d'observation - dimensions : 8
Espace d'action - dimensions : 4


### Espaces d'observation et d'actions

Voir https://gymnasium.farama.org/environments/box2d/lunar_lander/

#### _Espace d'observation_:

1. Coordonnées de l'alunisseur
   * x : position horizontale
   * y : position verticale
3. Vitesses linéraies
   * vx : vitesse horizontale
   * vy : vitesse verticale
5. Angle et vitesse angulaire
   * angle de l'alunisseur en radians
   * vitesse de rotation
7. Etat de contact avec le sol
   * Pied gauche touche le sol
   * Pied droit touche le sol

#### _Espace d'actions_: 

* 0 : pas d'action
* 1 : moteur gauche allumé
* 2 : moteur principal allumé
* 3 : moteur droit allumé



### Récompenses : 

Après chaque étape, une récompense est attribuée. La récompense totale d'un épisode correspond à la somme des récompenses attribuées pour toutes les étapes de cet épisode.

Pour chaque étape, la récompense :

* augmente/diminue à mesure que l'atterrisseur se rapproche/s'éloigne de la plate-forme d'atterrissage.

* augmente/diminue à mesure que l'atterrisseur ralentit/accélère.

* diminue à mesure que l'atterrisseur s'incline (angle non horizontal).

* est augmentée de 10 points pour chaque jambe en contact avec le sol.

* est diminuée de 0,03 point à chaque image où un moteur latéral est en marche.

* est diminuée de 0,3 point à chaque image où le moteur principal est en marche.

* L'épisode reçoit une récompense supplémentaire de -100 ou +100 points en cas de crash ou d'atterrissage en toute sécurité, respectivement.

* Un épisode est considéré comme une solution s'il obtient au moins 200 points.

## Premiers essais

In [2]:
mlflow.set_experiment("lunarlander-dqn")
model_name = "dqn_lunarlander"
NB_STEPS = 1000000

experiment = mlflow.get_experiment_by_name("lunarlander-dqn")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

hparams = dict(
    policy="MlpPolicy",
    learning_rate=0.001,          # Learning rate 0.001
    buffer_size=50000,            # Taille du buffer de replay
    learning_starts=1000,         # Commence à apprendre après 1000 steps
    target_update_interval=500,   # Met à jour le réseau cible tous les 500 steps
    exploration_fraction=0.1,     # Explore pendant 10% du temps total
    tensorboard_log=log_dir,
)

with mlflow.start_run(run_name=f"{model_name}_{NB_STEPS}_steps"):
    mlflow.log_param("nb_steps", NB_STEPS)
    mlflow.log_params(hparams)
    
    
    model = DQN(
        policy=hparams["policy"],
        env=train_env,
        verbose=0,
        seed=SEED,
        learning_rate=hparams["learning_rate"],
        buffer_size=hparams["buffer_size"],
        learning_starts=hparams["learning_starts"],
        target_update_interval=hparams["target_update_interval"],
        exploration_fraction=hparams["exploration_fraction"],
        tensorboard_log=hparams["tensorboard_log"],
    )
    
    model.learn(total_timesteps=NB_STEPS)
    model_path = f"{model_name}.zip"
    model.save(model_name)  # SB3 ajoute .zip
    if os.path.exists(model_path):
        mlflow.log_artifact(model_path, artifact_path="model")
    
    mean_reward, std_reward = evaluate_policy(
        model, 
        video_env, 
        n_eval_episodes=50,
        warn=False  # évite les warnings
    )
    
    video_env.close()
    print(f"Récompense moyenne sur 50 épisodes : {mean_reward:.2f} +/- {std_reward:.2f}")
    mlflow.log_metric("eval_mean_reward", float(mean_reward))
    mlflow.log_metric("eval_std_reward", float(std_reward))
    mlflow.end_run()

2026/01/27 14:44:07 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Experiment ID: 619582485636765788
Tracking URI: http://mlflow:5000
Récompense moyenne sur 50 épisodes : 175.83 +/- 94.01
🏃 View run dqn_lunarlander_1000000_steps at: http://mlflow:5000/#/experiments/619582485636765788/runs/1b14e7a963354a56990a39b90c1c8956
🧪 View experiment at: http://mlflow:5000/#/experiments/619582485636765788
